# Credit Risk Pipeline - End-to-End ML Project

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/genieincodebottle/credit-risk-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/genieincodebottle/aiml-companion/blob/main/projects/credit-risk-pipeline/notebooks/credit-risk-pipeline.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-View_Source-blue?logo=github)](https://github.com/genieincodebottle/aiml-companion/tree/main/projects/credit-risk-pipeline)

**Author:** [genieincodebottle](https://github.com/genieincodebottle) | **Dataset:** [German Credit (UCI/OpenML)](https://www.openml.org/d/31) | **Last Updated:** March 2026

<div class="alert alert-info">
<strong>Objective:</strong> Build a production-grade ML pipeline for credit default prediction.<br>
<strong>Dataset:</strong> German Credit - 1000 borrowers, 20 features<br>
<strong>Time:</strong> ~20-30 minutes | <strong>Difficulty:</strong> Intermediate
</div>

---

<a id="toc"></a>
## Table of Contents

1. [Setup & Imports](#setup) - Install dependencies and import libraries
2. [Data Loading](#data-loading) - Load German Credit dataset
3. [Data Quality Assessment](#data-quality) - Missing values, duplicates, target distribution
4. [Exploratory Data Analysis](#eda) - Correlations, distributions by target
5. [Feature Engineering](#feature-eng) - Monthly burden, age groups, log transforms
6. [Preprocessing Pipeline](#preprocessing) - ColumnTransformer with imputer + scaler + encoder
7. [Model Training (5-Fold CV)](#model-training) - Logistic Regression vs Gradient Boosting
8. [Cost-Sensitive Threshold Tuning](#threshold) - Optimize for FN:FP = 10:1 cost ratio
9. [SHAP Explanations](#shap) - Feature importance for adverse action reasons
10. [PSI Drift Monitoring](#psi) - Production drift detection
11. [Summary](#summary) - Key findings and production checklist

---

<a id="setup"></a>
## 1. Setup & Imports

In [ ]:
# Install dependencies (run this cell first on Kaggle/Colab)
!pip install -q shap scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

from sklearn.datasets import fetch_openml
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve

print("All imports successful!")

---

<a id="data-loading"></a>
## 2. Data Loading

Loading the German Credit dataset from OpenML (with CSV fallback). Each row represents a borrower with 20 features describing their financial profile.

In [ ]:
try:
    data = fetch_openml(data_id=31, as_frame=True, parser="auto")
    df = data.frame.copy()
except Exception as e:
    print(f"OpenML fetch failed ({e}). Falling back to CSV URL...")
    url = "https://raw.githubusercontent.com/genieincodebottle/aiml-companion/main/projects/credit-risk-pipeline/data/german_credit.csv"
    try:
        df = pd.read_csv(url)
    except Exception:
        raise RuntimeError(
            "Could not load data from OpenML or fallback URL.\n"
            "Manual fix: download German Credit from https://www.openml.org/d/31 "
            "and place as 'german_credit.csv' in the notebook directory, then use:\n"
            "  df = pd.read_csv('german_credit.csv')"
        )

if "class" in df.columns:
    df["target"] = (df["class"] == "bad").astype(int)
    df = df.drop(columns=["class"])

print(f"Shape: {df.shape}")
print(f"Default rate: {df['target'].mean():.1%}")
df.head()

In [ ]:
df.info()

<div class="alert alert-success">
<strong>Data loaded.</strong> The German Credit dataset contains 1000 borrowers with 20 features (mix of numeric and categorical) and a binary target (good/bad credit risk).
</div>

---

<a id="data-quality"></a>
## 3. Data Quality Assessment

Check for missing values, duplicates, and class imbalance before any modeling.

In [ ]:
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print(f"\nTarget distribution:")
print(df["target"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["target"].value_counts()
ax.bar(["No Default (0)", "Default (1)"], counts.values, color=["#22c55e", "#ef4444"])
for i, c in enumerate(counts.values):
    ax.text(i, c + 10, f"{c} ({c/len(df):.1%})", ha="center")
ax.set_title("Target Distribution")
plt.tight_layout()
plt.show()

<div class="alert alert-success">
<strong>Data Quality Summary:</strong>
<ul>
<li><strong>Class imbalance:</strong> ~30% default rate - moderate imbalance, manageable with class weights</li>
<li>Check for missing values and duplicates before proceeding</li>
<li>The 70/30 split is important - it means accuracy alone is misleading (a "predict no default" baseline gets 70%)</li>
</ul>
</div>

---

<a id="eda"></a>
## 4. Exploratory Data Analysis

Visualize feature correlations and distributions segmented by default status.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != "target"]
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"Numeric ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
corr = df[numeric_cols + ["target"]].corr()
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, square=True)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
n = min(len(numeric_cols), 6)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, col in enumerate(numeric_cols[:n]):
    ax = axes[i//3][i%3]
    for val, color, label in [(0, "#22c55e", "No Default"), (1, "#ef4444", "Default")]:
        ax.hist(df[df["target"]==val][col].dropna(), bins=25, alpha=0.5, color=color, label=label, density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=7)
for i in range(n, 6):
    axes[i//3][i%3].set_visible(False)
plt.suptitle("Feature Distributions by Target", fontsize=13)
plt.tight_layout()
plt.show()

<div class="alert alert-info">
<strong>EDA Insights:</strong>
<ul>
<li>The correlation heatmap reveals which numeric features co-vary - watch for multicollinearity</li>
<li>Feature distributions split by target show where defaulters differ from non-defaulters</li>
<li>Features with clear separation between classes (e.g., duration, credit amount) are likely strong predictors</li>
</ul>
</div>

---

<a id="feature-eng"></a>
## 5. Feature Engineering

Create domain-specific features that capture borrower risk more effectively than raw columns alone.

<div class="alert alert-info">
<strong>Features created:</strong>
<ul>
<li><strong>Monthly burden</strong> - Credit amount / duration (captures payment pressure)</li>
<li><strong>Loan burden</strong> - Normalized credit x duration product</li>
<li><strong>Age groups</strong> - Bucketed age for non-linear risk patterns</li>
<li><strong>Log transforms</strong> - Applied to highly skewed features (skew > 2.0)</li>
</ul>
</div>

In [ ]:
if "credit_amount" in df.columns and "duration" in df.columns:
    df["monthly_burden"] = df["credit_amount"] / df["duration"].replace(0, np.nan)
    df["loan_burden"] = (df["credit_amount"] * df["duration"]) / df["credit_amount"].median()
    print("Added monthly_burden, loan_burden")

if "age" in df.columns:
    df["age_group"] = pd.cut(df["age"], bins=[0,25,35,45,55,100], labels=["18-25","26-35","36-45","46-55","55+"])
    print(f"Age groups: {df['age_group'].value_counts().sort_index().to_dict()}")

for col in numeric_cols:
    if abs(df[col].skew()) > 2.0:
        df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))
        print(f"Added log_{col} (skew={df[col].skew():.1f})")

print(f"\nTotal features: {df.shape[1]}")

<div class="alert alert-success">
<strong>Features Added:</strong>
<ul>
<li><strong>monthly_burden</strong> and <strong>loan_burden</strong> capture payment stress better than raw credit amount alone</li>
<li><strong>age_group</strong> allows the model to learn non-linear age-risk patterns (young and very old borrowers tend to have higher risk)</li>
<li><strong>Log transforms</strong> reduce the influence of extreme outliers in skewed features</li>
</ul>
</div>

---

<a id="preprocessing"></a>
## 6. Preprocessing Pipeline

Build a leakage-proof `ColumnTransformer` that handles numeric and categorical features separately.

<div class="alert alert-warning">
<strong>Why Pipeline matters:</strong> Wrapping preprocessing + model in a single Pipeline ensures that scaling and encoding are fitted ONLY on training folds during cross-validation. This prevents data leakage from the test set.
</div>

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", KNNImputer(n_neighbors=5)), ("scaler", StandardScaler())]), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
], remainder="drop")

print(f"Numeric: {len(numeric_features)} features -> KNNImputer -> StandardScaler")
print(f"Categorical: {len(categorical_features)} features -> OneHotEncoder")

---

<a id="model-training"></a>
## 7. Model Training (5-Fold CV)

Compare two models:
- **Logistic Regression** (with `class_weight="balanced"`) - interpretable baseline
- **Gradient Boosting** - more powerful but less transparent

<div class="alert alert-info">
<strong>Metrics:</strong> ROC-AUC (ranking quality), Recall (catch defaults), F1 (balanced precision-recall)
</div>

In [ ]:
lr_pipe = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=42))])
gbc_pipe = Pipeline([("pre", preprocessor), ("clf", GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=4, subsample=0.8, random_state=42))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}
for name, pipe in [("LogisticRegression", lr_pipe), ("GradientBoosting", gbc_pipe)]:
    print(f"\nTraining {name}...")
    scores = cross_validate(pipe, X, y, cv=cv, scoring=["roc_auc", "recall", "f1"])
    results[name] = {k: scores[f"test_{k}"].mean() for k in ["roc_auc", "recall", "f1"]}
    results[name]["roc_auc_std"] = scores["test_roc_auc"].std()
    print(f"  ROC-AUC: {results[name]['roc_auc']:.3f} (+/- {results[name]['roc_auc_std']:.3f})")
    print(f"  Recall:  {results[name]['recall']:.3f}")
    print(f"  F1:      {results[name]['f1']:.3f}")

In [ ]:
---

<a id="threshold"></a>
## 8. Cost-Sensitive Threshold Tuning

<div class="alert alert-danger">
<strong>Business context:</strong> A missed default (FN) costs <strong>$10</strong> in losses, while a wrongly denied loan (FP) costs only <strong>$1</strong> in lost revenue. The default 0.5 threshold does not account for this asymmetry.
</div>

We sweep thresholds from 0.05 to 0.95 and pick the one that minimizes total cost.

> **Note**: For simplicity, we fit on the full dataset and tune the threshold here. In production, always tune the threshold on a held-out validation set or use nested cross-validation to avoid overfitting the threshold to training data.

## 8. Cost-Sensitive Threshold Tuning

FN cost = $10 (missed default), FP cost = $1 (denied good borrower)

> **Note**: For simplicity, we fit on the full dataset and tune the threshold here. In production, always tune the threshold on a held-out validation set or use nested cross-validation to avoid overfitting the threshold to training data.

<div class="alert alert-success">
<strong>Model Comparison Results:</strong>
<ul>
<li><strong>Gradient Boosting</strong> typically achieves higher ROC-AUC than Logistic Regression on this dataset</li>
<li><strong>Logistic Regression</strong> with balanced class weights achieves competitive recall</li>
<li>The performance gap is moderate - both models are reasonable choices depending on interpretability requirements</li>
<li>Cross-validation standard deviations show how stable the estimates are across folds</li>
</ul>
</div>

In [ ]:
best_name = max(results, key=lambda k: results[k]["roc_auc"])
best_pipe = lr_pipe if best_name == "LogisticRegression" else gbc_pipe
best_pipe.fit(X, y)
y_proba = best_pipe.predict_proba(X)[:, 1]

COST_FN, COST_FP = 10, 1
thresholds = np.arange(0.05, 0.95, 0.01)
costs = []
for t in thresholds:
    cm = confusion_matrix(y, (y_proba>=t).astype(int))
    costs.append(cm[1,0]*COST_FN + cm[0,1]*COST_FP)

opt_t = thresholds[np.argmin(costs)]
min_cost = min(costs)
print(f"Best model: {best_name}")
print(f"Optimal threshold: {opt_t:.2f}")
print(f"Minimum cost: ${min_cost:,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(thresholds, costs, "b-", lw=2)
axes[0].axvline(x=opt_t, color="red", linestyle="--", label=f"Optimal: {opt_t:.2f}")
axes[0].axvline(x=0.5, color="gray", linestyle=":", label="Default: 0.50")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Total Cost ($)")
axes[0].set_title("Cost-Sensitive Threshold Tuning")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

precision, recall, pr_t = precision_recall_curve(y, y_proba)
axes[1].plot(recall, precision, "b-", lw=2)
idx = np.argmin(np.abs(pr_t - opt_t))
axes[1].plot(recall[idx], precision[idx], "ro", markersize=12, label=f"Optimal (t={opt_t:.2f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<div class="alert alert-success">
<strong>Threshold Tuning Results:</strong>
<ul>
<li>The optimal threshold is typically <strong>lower than 0.50</strong> (around 0.20-0.35) due to the 10:1 cost asymmetry</li>
<li>This catches more defaults (higher recall) at the expense of more false alarms (lower precision)</li>
<li>The cost curve shows a clear minimum - thresholds above 0.5 miss too many defaults, below 0.1 deny too many good borrowers</li>
<li>The confusion matrix comparison shows the tradeoff: fewer FNs (costly) but more FPs (cheap)</li>
</ul>
</div>

In [ ]:
---

<a id="shap"></a>
## 9. SHAP Explanations

<div class="alert alert-info">
<strong>Why SHAP for credit risk?</strong> Regulatory requirements (e.g., ECOA, FCRA) mandate that lenders provide <strong>adverse action reasons</strong> when denying credit. SHAP values identify which features most contributed to a denial decision, enabling compliant explanations like "Your credit amount relative to duration is too high."
</div>

## 9. SHAP Explanations

<div class="alert alert-success">
<strong>SHAP Results:</strong>
<ul>
<li>The top features driving default predictions are typically related to credit amount, duration, and account status</li>
<li>These align with domain knowledge - larger loans over longer periods with poor account history increase default risk</li>
<li>SHAP provides per-prediction explanations, not just global importance - essential for individual adverse action notices</li>
</ul>
</div>

In [ ]:
---

<a id="psi"></a>
## 10. PSI Drift Monitoring

<div class="alert alert-warning">
<strong>What is PSI?</strong> Population Stability Index measures how much the distribution of a feature has shifted between reference (training) and production data.
<ul>
<li><strong>PSI < 0.10</strong> - No significant shift (OK)</li>
<li><strong>PSI 0.10-0.25</strong> - Moderate shift (WARN - investigate)</li>
<li><strong>PSI > 0.25</strong> - Major shift (ALERT - retrain model)</li>
</ul>
</div>

We simulate production drift by inflating `credit_amount` to demonstrate detection.

## 10. PSI Drift Monitoring

<div class="alert alert-success">
<strong>Key Observations:</strong>
<ul>
<li>Most features show PSI < 0.10 (no drift) since production data was sampled from the same distribution</li>
<li><code>credit_amount</code> shows high PSI (ALERT) because we intentionally inflated it by 1.5x + 500</li>
<li>In production, an ALERT on a key feature like credit amount would trigger model retraining</li>
</ul>
</div>

In [ ]:
---

<a id="summary"></a>
## 11. Summary

<div class="alert alert-success">
<strong>What we built:</strong>
<ol>
<li><strong>Data pipeline</strong> - German Credit loaded, cleaned, validated</li>
<li><strong>EDA</strong> - Distributions, correlations, target segmentation</li>
<li><strong>Feature engineering</strong> - Monthly burden, loan burden, age buckets, log transforms</li>
<li><strong>Model training</strong> - LR + GBC with sklearn Pipeline (leakage-proof)</li>
<li><strong>Cost-sensitive evaluation</strong> - 10:1 FN/FP threshold tuning</li>
<li><strong>SHAP explanations</strong> - Feature importance for adverse action reasons</li>
<li><strong>PSI monitoring</strong> - Drift detection for production</li>
</ol>
</div>

<div class="alert alert-info">
<strong>Production checklist:</strong>
<ul>
<li>Wrap preprocessing + model in a single Pipeline to prevent leakage</li>
<li>Tune threshold on held-out validation set (not training data)</li>
<li>Monitor PSI weekly and trigger retraining when PSI > 0.25</li>
<li>Log SHAP values per prediction for audit trail</li>
</ul>
</div>

> Full project: [github.com/genieincodebottle/aiml-companion/projects/credit-risk-pipeline](https://github.com/genieincodebottle/aiml-companion/tree/main/projects/credit-risk-pipeline)

---

<div class="alert alert-info">
<strong>Found this notebook useful?</strong> Give it an upvote on Kaggle! Have questions or suggestions? Leave a comment below or open an issue on <a href="https://github.com/genieincodebottle/aiml-companion">GitHub</a>.
</div>

## Summary

1. **Data pipeline** - German Credit loaded, cleaned, validated
2. **EDA** - Distributions, correlations, target segmentation
3. **Feature engineering** - Monthly burden, loan burden, age buckets, log transforms
4. **Model training** - LR + GBC with sklearn Pipeline (leakage-proof)
5. **Cost-sensitive evaluation** - 10:1 FN/FP threshold tuning
6. **SHAP explanations** - Feature importance for adverse action reasons
7. **PSI monitoring** - Drift detection for production

> Full project: [github.com/genieincodebottle/aiml-companion/projects/credit-risk-pipeline](https://github.com/genieincodebottle/aiml-companion/tree/main/projects/credit-risk-pipeline)